# Person B – Image size experiments (320, 480, 800) + evaluation on our own photos
Baseline (from `Vegetable_Classification_Baseline.ipynb`): `yolo11n-cls.pt`, epochs=15, imgsz=224, **batch=16**, seed=42.
This notebook changes **only image size** (320, 480, 800), then tests the baseline and all 3 runs on the group's own 120 photos (the assignment's test set).

Runtime → Change runtime type → **T4 GPU** before running. Run cells top to bottom.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q ultralytics

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Settings – check these two paths

In [ ]:
PROJECT = "/content/drive/MyDrive/IAT 360 CV PROJECT"        # same folder the baseline used
ZIP_PATH = f"{PROJECT}/vegetable_dataset.zip"
OWN_PHOTOS_ROOT = PROJECT   # folder that contains everyone's photo folders (Alyssa, etc.). Searched recursively.
RESULTS = f"{PROJECT}/training_results"
BASELINE_WEIGHTS = f"{RESULTS}/baseline/weights/best.pt"

CLASSES = ["Broccoli", "Capsicum", "Carrot", "Potato", "Tomato"]   # must match the dataset folder names
BASE = dict(epochs=15, batch=16, seed=42)                          # copied from the baseline – do not change
IMGSZ_SIZES = [320, 480, 800]                                           # baseline was 224

## 1. Unzip the online dataset

In [8]:
import zipfile, os
with zipfile.ZipFile(ZIP_PATH) as z: z.extractall("/content/")
DATA = "/content/vegetable_dataset"
for s in ["train","val","test"]:
    print(s, {c: len(os.listdir(f"{DATA}/{s}/{c}")) for c in sorted(os.listdir(f"{DATA}/{s}")) if os.path.isdir(f"{DATA}/{s}/{c}")})

train {'Broccoli': 1000, 'Capsicum': 1000, 'Carrot': 1000, 'Potato': 1000, 'Tomato': 1000}
val {'Broccoli': 200, 'Capsicum': 200, 'Carrot': 200, 'Potato': 200, 'Tomato': 200}
test {'Broccoli': 200, 'Capsicum': 200, 'Carrot': 200, 'Potato': 200, 'Tomato': 200}


## 2. Build the own-photo test set
Finds every photo named like `carrot_alyssa_003.jpeg` anywhere under `OWN_PHOTOS_ROOT` and sorts it into a class folder by the first word of its name.
Check the printed counts: you want ~24 per class (6 × 4 people) and 120 total. Anything listed as *skipped* has a name that didn't match – rename it in Drive and rerun.

In [9]:
import re, shutil, glob
from collections import Counter
OWN = "/content/own_test_ds"
shutil.rmtree(OWN, ignore_errors=True)
for c in CLASSES: os.makedirs(f"{OWN}/test/{c}", exist_ok=True)
os.symlink(f"{DATA}/train", f"{OWN}/train"); os.symlink(f"{DATA}/val", f"{OWN}/val")   # ultralytics wants these to exist

lookup = {c.lower(): c for c in CLASSES}
lookup.update({"pepper": "Capsicum", "bellpepper": "Capsicum", "tomatoes": "Tomato", "carrots": "Carrot", "potatoes": "Potato"})
pat = re.compile(r"^([a-z]+)_([a-z]+)_(\d+)", re.I)
counts, people, skipped, seen = Counter(), Counter(), [], set()
for p in glob.glob(f"{OWN_PHOTOS_ROOT}/**/*", recursive=True):
    if "vegetable_dataset" in p or "training_results" in p or not os.path.isfile(p): continue
    if not p.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
        if p.lower().endswith((".heic", ".heif")): skipped.append(p + "   <- HEIC: convert to .jpg")
        continue
    name = os.path.basename(p); m = pat.match(name)
    if not m or m.group(1).lower() not in lookup: skipped.append(p); continue
    key = (m.group(1).lower(), m.group(2).lower(), int(m.group(3)))
    if key in seen: skipped.append(p + "   <- duplicate"); continue
    seen.add(key); cls = lookup[m.group(1).lower()]
    shutil.copy(p, f"{OWN}/test/{cls}/{m.group(1).lower()}_{m.group(2).lower()}_{int(m.group(3)):03d}.jpg")
    counts[cls] += 1; people[m.group(2).lower()] += 1
print("Per class:", dict(counts)); print("Per person:", dict(people)); print("Total:", sum(counts.values()))
print("Skipped:", *skipped[:40], sep="\n  ")

Per class: {'Potato': 24, 'Tomato': 24, 'Capsicum': 24, 'Carrot': 24, 'Broccoli': 24}
Per person: {'sumerah': 30, 'alyssa': 30, 'grace': 30, 'antony': 30}
Total: 120
Skipped:
  /content/drive/MyDrive/IAT 360 CV PROJECT/original photos/Broccoli/broccoli_sumerah_001.jpeg   <- duplicate
  /content/drive/MyDrive/IAT 360 CV PROJECT/original photos/Broccoli/broccoli_sumerah_003.jpeg   <- duplicate
  /content/drive/MyDrive/IAT 360 CV PROJECT/original photos/Broccoli/broccoli_sumerah_004.jpeg   <- duplicate
  /content/drive/MyDrive/IAT 360 CV PROJECT/original photos/Broccoli/broccoli_sumerah_002.jpeg   <- duplicate
  /content/drive/MyDrive/IAT 360 CV PROJECT/original photos/Broccoli/broccoli_sumerah_005.jpeg   <- duplicate
  /content/drive/MyDrive/IAT 360 CV PROJECT/original photos/Broccoli/broccoli_sumerah_006.jpeg   <- duplicate
  /content/drive/MyDrive/IAT 360 CV PROJECT/original photos/Broccoli/broccoli_alyssa_003.jpeg   <- duplicate
  /content/drive/MyDrive/IAT 360 CV PROJECT/original pho

[link text](https://)## 3. Train the 3 images-size runs (~10 min each on a T4)
Everything except `image` is identical to the baseline. Results save to Drive automatically.

In [10]:
from ultralytics import YOLO

for sz in IMGSZ_SIZES:
    run_name = f"grace_imgsz_{sz}"
    if os.path.exists(f"{RESULTS}/{run_name}/weights/best.pt"):
        print(f"{run_name} already trained — skipping"); continue
    YOLO("yolo11n-cls.pt").train(data=DATA, imgsz=sz, project=RESULTS, name=run_name,
                                 exist_ok=True, plots=True, **BASE)

Ultralytics 8.4.204 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/vegetable_dataset, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=grace_imgsz_320, nbs=64, nms=None, opset=None, optimize=Fals

## 4. Evaluate baseline + all runs on (a) the online test split and (b) our own photos
(b) is the number that matters for the report – it's the assignment's test set.

In [11]:
import pandas as pd
runs = {"baseline (imgsz=224)": (BASELINE_WEIGHTS, 224),
        **{f"imgsz {sz}": (f"{RESULTS}/grace_imgsz_{sz}/weights/best.pt", sz) for sz in IMGSZ_SIZES}}
rows = []
for label, (w, sz) in runs.items():
    m = YOLO(w)
    online = m.val(data=DATA, split="test", imgsz=sz, batch=32, project=RESULTS, name=f"eval_online_{sz}", exist_ok=True)
    own = m.val(data=OWN, split="test", imgsz=sz, batch=32, project=RESULTS, name=f"eval_own_{sz}", exist_ok=True)
    rows.append({"run": label, "online test top1": round(online.top1, 4), "online test top5": round(online.top5, 4),
                 "OWN photos top1": round(own.top1, 4), "OWN photos top5": round(own.top5, 4)})
table = pd.DataFrame(rows); table.to_csv(f"{RESULTS}/imgsz_comparison.csv", index=False)
table

Ultralytics 8.4.204 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n-cls summary (fused): 47 layers, 1,532,429 parameters, 0 gradients, 3.2 GFLOPs
train: /content/vegetable_dataset/train... found 5000 images in 5/5 classes ✅ 
val: /content/vegetable_dataset/val... found 1000 images in 5/5 classes ✅ 
test: /content/vegetable_dataset/test... found 1000 images in 5/5 classes ✅ 
WARNING ⚠️ test: Slow image access detected (ping: 0.0±0.0 ms, read: 23.9±23.3 MB/s, size: 44.3 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
test: Scanning /content/vegetable_dataset/test... 1000 images, 0 corrupt: 100% ━━━━━━━━━━━━ 1000/1000 1.2Kit/s 0.8s
test: New cache created: /content/vegetable_dataset/test.cache
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 32/32 10.8it/s 3.0s
                   all          1          1
Speed: 0.1ms preprocess, 1.3ms inference, 0.0ms loss, 

,run,online test top1,online test top5,OWN photos top1,OWN photos top5
0,baseline (imgsz=224),1.0,1.0,0.7333,1.0
1,imgsz 320,1.0,1.0,0.7583,1.0
2,imgsz 480,1.0,1.0,0.6583,1.0
3,imgsz 800,1.0,1.0,0.6000,1.0


## 5. Per-class accuracy on own photos + list of mistakes (for the report)

In [12]:
from collections import defaultdict
for label, (w, sz) in runs.items():
    m = YOLO(w); correct = defaultdict(int); total = defaultdict(int); wrong = []
    for c in CLASSES:
        for f in sorted(glob.glob(f"{OWN}/test/{c}/*")):
            r = m.predict(f, imgsz=sz, verbose=False)[0]
            pred = r.names[r.probs.top1]; total[c] += 1
            if pred == c: correct[c] += 1
            else: wrong.append((os.path.basename(f), c, pred, round(float(r.probs.top1conf), 2)))
    print(f"\n=== {label} ===")
    print({c: f"{correct[c]}/{total[c]}" for c in CLASSES})
    for x in wrong: print("  WRONG:", x[0], "| true:", x[1], "| predicted:", x[2], "| conf:", x[3])


=== baseline (imgsz=224) ===
{'Broccoli': '24/24', 'Capsicum': '20/24', 'Carrot': '14/24', 'Potato': '15/24', 'Tomato': '15/24'}
  WRONG: capsicum_antony_003.jpg | true: Capsicum | predicted: Carrot | conf: 0.91
  WRONG: capsicum_antony_004.jpg | true: Capsicum | predicted: Broccoli | conf: 0.55
  WRONG: capsicum_sumerah_005.jpg | true: Capsicum | predicted: Broccoli | conf: 0.85
  WRONG: capsicum_sumerah_006.jpg | true: Capsicum | predicted: Potato | conf: 0.74
  WRONG: carrot_alyssa_001.jpg | true: Carrot | predicted: Tomato | conf: 1.0
  WRONG: carrot_alyssa_002.jpg | true: Carrot | predicted: Broccoli | conf: 0.43
  WRONG: carrot_alyssa_005.jpg | true: Carrot | predicted: Tomato | conf: 0.99
  WRONG: carrot_grace_001.jpg | true: Carrot | predicted: Broccoli | conf: 0.98
  WRONG: carrot_grace_002.jpg | true: Carrot | predicted: Broccoli | conf: 0.99
  WRONG: carrot_grace_003.jpg | true: Carrot | predicted: Broccoli | conf: 0.94
  WRONG: carrot_grace_004.jpg | true: Carrot | predict

## 6. Where your report figures are (in Drive → `training_results/`)
-  `results.png` (training curves), `confusion_matrix_normalized.png`, `results.csv`
- `eval_own_*/` → confusion matrix on our own photos (most useful figure for the shortcomings section)
- `image_size_comparison.csv` → the comparison table